In [1]:
# # SB3 PPO Model Visualization and Testing
# This notebook allows you to load the trained models from the hyperparameter 
# tuning experiments and visualize their performance in the environment.
import os
import time
import gymnasium as gym
from stable_baselines3 import PPO
# Import configuration to ensure consistency with training
import config 

In [2]:
# 1. Define the directory where models are saved and list them
models_dir = "results/models"

if os.path.exists(models_dir):
    available_models = [f for f in os.listdir(models_dir) if f.endswith(".zip")]
    
    if available_models:
        print("Available trained models:")
        for i, model_name in enumerate(available_models):
            print(f"[{i}] {model_name}")
    else:
        print(f"No '.zip' models found in {models_dir}.")
else:
    print(f"Directory '{models_dir}' not found. Please run the training script first.")
    available_models = []

Available trained models:
[0] model_lr0.0003_g0.99_gae0.95_clip0.2_arch128_128_ep480_rew168.06.zip
[1] model_lr0.0003_g0.99_gae0.95_clip0.2_arch128_128_ep500_rew171.56.zip
[2] model_lr0.0003_g0.99_gae0.95_clip0.2_arch128_128_ep520_rew200.19.zip


In [3]:
# 2. Select and load a specific model
# Change this index to select a different model from the list above
MODEL_INDEX = -1 # Default to the last one in the list

if available_models:
    selected_model_name = available_models[MODEL_INDEX]
    selected_model_path = os.path.join(models_dir, selected_model_name)
    
    print(f"Loading model: {selected_model_name}")
    
    # Load the trained model
    model = PPO.load(selected_model_path)
    print("Model loaded successfully!")
else:
    print("Cannot proceed: No models to load.")

Loading model: model_lr0.0003_g0.99_gae0.95_clip0.2_arch128_128_ep520_rew200.19.zip
Model loaded successfully!


d:\Anaconda3\envs\script\lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [7]:
# 3. Visualize the trained agent in the environment
# We use render_mode="human" to open a local window and watch the agent play
model = PPO.load("results/models/model_lr0.0003_g0.99_gae0.95_clip0.2_arch128_128_ep480_rew168.06.zip", device="cpu")

# Create the environment with human rendering enabled
env = gym.make(
    config.ENV_NAME, 
    continuous=config.IS_CONTINUOUS, 
    render_mode="human"
)

# Number of test episodes to visualize
test_episodes = 3

for ep in range(test_episodes):
    obs, info = env.reset()
    done = False
    truncated = False
    episode_reward = 0.0
    
    while not (done or truncated):
        # Predict the action using the trained model
        # Setting deterministic=True typically yields better and more stable test performance
        action, _states = model.predict(obs, deterministic=True)
        
        # Step the environment forward
        obs, reward, done, truncated, info = env.step(action)
        episode_reward += reward

        env.render()
        
        # Optional: Add a slight delay if the rendering is too fast
        time.sleep(0.01)
        
    print(f"Test Episode {ep + 1} | Total Reward: {episode_reward:.2f}")
    
# Close the environment rendering window
env.close()
print("Visualization completed.")

Test Episode 1 | Total Reward: 234.92
Test Episode 2 | Total Reward: 170.04
Test Episode 3 | Total Reward: -54.88
Visualization completed.
